# Task 5 — Airline passengers: trend and seasonality

**Week 1 data visualization assignment**

This notebook follows the task brief and chart-design rules: titles state findings, bars use a zero baseline, color carries meaning, excluded rows are stated, and each chart has a short rationale.

## Setup and data

Run the cells from top to bottom. The CSV is read from its public source URL; an internet connection is required. If needed, install `pandas`, `numpy`, and `matplotlib` in your notebook environment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
DATA_URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv'
df = pd.read_csv(DATA_URL)
df.head()

## Create an ordered monthly time axis

In [ ]:
month_num=pd.to_datetime(df.month,format='%B').dt.month
df['date']=pd.to_datetime(dict(year=df.year,month=month_num,day=1))
df=df.sort_values('date')
df.head()

## 1. Overall monthly trend

In [ ]:
fig,ax=plt.subplots(figsize=(11,4.5)); ax.plot(df.date,df.passengers,color='#0072B2',linewidth=1.8)
ax.set(title='Monthly passenger totals rose sharply from 1949 to 1960',xlabel='Year',ylabel='Passengers (thousands)'); fig.tight_layout()

**Why a line chart:** monthly observations are ordered in time; connecting them reveals both the long-term rise and recurring seasonal cycle.

## 2. Isolate the seasonal cycle

In [ ]:
months=list(pd.date_range('2000-01-01',periods=12,freq='MS').strftime('%B'))
fig,ax=plt.subplots(figsize=(10,5))
for year,g in df.groupby('year'):
    g=g.assign(month_num=pd.to_datetime(g.month,format='%B').dt.month).sort_values('month_num')
    ax.plot(g.month_num,g.passengers,marker='o',alpha=.75,label=str(year))
ax.set_xticks(range(1,13),months,rotation=35,ha='right')
ax.set(title='Passenger totals peak in summer in every year',xlabel='Month',ylabel='Passengers (thousands)')
ax.legend(title='Year',ncol=3,fontsize=8,frameon=False); fig.tight_layout()

**Why this chart:** one line per year aligns the months and makes the recurring within-year shape comparable across years. The line chart emphasizes the seasonal profile, while the first chart makes the full time trend clearest.

## 3. Which month is busiest?

In [ ]:
monthly=df.groupby('month').passengers.mean().reindex(months)
fig,ax=plt.subplots(figsize=(9,4.5)); ax.bar(monthly.index,monthly.values,color=['#B8C4CC' if m!='July' else '#D55E00' for m in monthly.index])
ax.set(title='July has the highest average passenger total',xlabel='Month',ylabel='Mean passengers (thousands)'); ax.set_ylim(bottom=0); plt.xticks(rotation=35,ha='right'); fig.tight_layout()
monthly.sort_values(ascending=False).head()

**Why a bar chart:** the 12 months are discrete categories and their mean totals are directly comparable; the zero baseline preserves magnitude. July is highlighted because it is the finding, while other bars stay neutral.

## Report choice and growing seasonal swing

For a report about overall growth, I would use the first chart because it shows the complete monthly history in one view. The seasonal small multiples and monthly average answer narrower questions; putting all three together would repeat some information and compete for attention.

The larger absolute seasonal swing could reflect the overall growth in passenger volume rather than stronger seasonality. To distinguish these explanations, plot a relative seasonal measure for each year—for example, each month's passengers divided by that year's mean (or the seasonal amplitude divided by that year's mean). If the relative seasonal profile grows over time, seasonality itself is strengthening; if it stays similar, the larger swing is mainly scale.